### 🎯 Module Overview:
- This module covers everything we need to know about parsing and ingesting data for RAG systems, from basic text files to complex PDFs and databases. We'll use LangChain v0.3 and explore each technique with practical examples
- Table of Contents:
    - Introduction to Data Ingestion
    - Text Files (.txt)
    - PDF Documents
    - Microsoft Word Documents
    - CSV and Excel Files
    - Json and Structured Data
    - Web Scraping
    - Databases (SQL)
    - Audio and Video Transcripts
    - Advance Techniques
    - Best Practices

#### Introduction To Data Ingestion

In [1]:
import os
from typing import List, Dict, Any
import pandas as pd

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)
from langchain_text_splitters import TokenTextSplitter
print("Set-Up Completed")

Set-Up Completed


#### Understanding Document Structure in Langchain

In [3]:
# Create a simple document
doc = Document(
    page_content = "This is the main text content that will be embedded and searched.",
    metadata = {
        "source": "example.txt",
        "page":1,
        "author": "Mani and Ruchi",
        "date_created": "2026-03-28",
        "custom_field": "any_value"
    }
)

print("Document Structure")
print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

Document Structure
Content: This is the main text content that will be embedded and searched.
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'Mani and Ruchi', 'date_created': '2026-03-28', 'custom_field': 'any_value'}


Why Metadata Matters?... 
- Metadata is crucial for following:
    1. Filtering search results
    2. Tracking document sources
    3. Providing context in responses
    4. Debugging and auditing

In [4]:
type(doc)

langchain_core.documents.base.Document

#### Text Files (.txt) - The simplest Case {text-files}

In [5]:
# Create a simple text file
import os
os.makedirs("data/text_files", exist_ok=True)

In [6]:
sample_texts = {
    "data/text_files/python_introduction.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
"""
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample Text file is created")

Sample Text file is created


#### Text Loader - Read Single File

In [7]:
from langchain_community.document_loaders import TextLoader

# Loading a single text file
loader = TextLoader("data/text_files/python_introduction.txt", encoding="utf-8")
documents=loader.load()

print(f"Loaded {len(documents)} documents")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")

Loaded 1 documents
Content preview: Python Programming Introduction

Python is a high-level, interpreted programming language known for ...
Metadata: {'source': 'data/text_files/python_introduction.txt'}


#### Directory Loader: Multiple Text Files

In [8]:
from langchain_community.document_loaders import DirectoryLoader

# load all the text files from the directory
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files structure
    loader_cls=TextLoader,  ## Loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)

documents = dir_loader.load()
print(f"Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\n Document {i+1}:")
    print(f"Source: {doc.metadata['source']}")
    print(f"Length {len(doc.page_content)} characters")

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 142.78it/s]

Loaded 2 documents

 Document 1:
Source: data\text_files\machine_learning.txt
Length 569 characters

 Document 2:
Source: data\text_files\python_introduction.txt
Length 489 characters


Directory Loader Characteristics:
- Advantages:
    - Loads multiple files at once
    - Supports glob patterns
    - Progress tracking
    - Recursive directory scanning

- Disadvantages:
    - All files must be same type
    - Limited error handling per file
    - Can be memory intensive for large directories


#### Text Splitting Strategies

In [9]:
# Different text splitting strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n'), Document(metadata={'source': 'data\\text_files\\python_introduction.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogr

In [10]:
text = documents[0].page_content
text

'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n'

In [11]:
# Method 1: Character-based splitting
print("1. Character Text Splitter:")
char_splitter = CharacterTextSplitter(
    separator=" ",    # Split on newlines
    chunk_size=200,    # Max chunk size in characters
    chunk_overlap=20,   # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First Chunk: {char_chunks[0][:100]}...")

1. Character Text Splitter:
Created 3 chunks
First Chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


In [12]:
print(char_chunks[0])
print("--------------------")
print(char_chunks[1])

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
--------------------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:


In [13]:
# Method 1: Character-based splitting
print("1. Character Text Splitter:")
char_splitter = CharacterTextSplitter(
    separator="\n",    # Split on newlines
    chunk_size=200,    # Max chunk size in characters
    chunk_overlap=20,   # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First Chunk: {char_chunks[0][:100]}...")

1. Character Text Splitter:
Created 4 chunks
First Chunk: Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems...


In [14]:
print(char_chunks[0])
print("-------------")
print(char_chunks[1])
print("-------------")
print(char_chunks[2])

Machine Learning Basics
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
-------------
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.
Types of Machine Learning:
-------------
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties


In [15]:
# Method 2: Recursive Character Splitting (Recommended)
print("\n 2. Recursive Character Text Splitter")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "], # Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")


 2. Recursive Character Text Splitter
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


In [16]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])
print("------------------")
print(recursive_chunks[2])

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
-----------------
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:
------------------
Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


In [17]:
# Create text without natural break points
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three which is even longer than the others. This is sentence four. This is sentence five. This is sentence six."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # only split on spaces
    chunk_size=80,
    chunk_overlap=20, 
    length_function=len
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i+1}: '{chunks[i]}")
    print(f"Chunk {i+2}: '{chunks[i+1]}")
    print()


Simple text example - 4 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also
Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than

Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than
Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.

Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.
Chunk 4: 'is sentence five. This is sentence six.



In [18]:
# Method 3: Token-based Splitting
print("\n 3. Token Text Splitter")
token_splitter = TokenTextSplitter(
    chunk_size=50, # size in tokens (not characters)
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")


 3. Token Text Splitter
Created 3 chunks
First chunk: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables system...


### Comparision:
- Text Splitting Methods Comparision:
    - CharacterTextSplitter:
        - ✅ Simple and predictable
        - ✅ Good for Structured text
        - ❌ May break mid-sentence
        - Use When: Text has clear delimiters

    - RecursiveCharacterTextSplitter:
        - ✅ Respects text structure
        - ✅ Tries multiple separators
        - ✅ Best general-purpose splitter
        - ❌ Slightly more complex
        - Use when: Default choice for most texts

    - TokenTextSplitter:
        - ✅ Respects model token limits
        - ✅ More accurate for embeddings
        - ❌ Slower than character-based
        - Use when: Working with token-limited models